# Experiment Runner
Speaker / Listener × Logit / Choice × 모델 선택

**사용법**: Config 만 수정하여 사용

In [1]:
import csv
from pathlib import Path
from random import sample
from variables import *
from scoring import score_candidates

In [2]:
# ── CONFIG ──────────────────────── (여기만 수정)
MODEL       = "qwen3-8b"   # "llama3" | "qwen3"
ROLE        = "listener"  # "speaker" | "listener"
MODE        = "choice"    # "logit"   | "choice"
REPETITIONS = 10          # logit → 1 권장 (결정론적),  choice → 5~10
SHOT = "zero"  # "zero" | "one" | "few"
# ────────────────────────────────────────────────

print(f"model={MODEL}  role={ROLE}  mode={MODE}  reps={REPETITIONS}")

model=qwen3-8b  role=listener  mode=choice  reps=10


In [3]:
# ── 모델 로드 ─────────────────────────────────────
from llama_cpp import Llama

llm = Llama.from_pretrained(
    **MODEL_CONFIGS[MODEL],
    n_ctx=2048,
    n_threads=4,
    logits_all=True,
)
print(f"로드 완료: {MODEL}")

/Users/eyun/Pragmatic_LM/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.025 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple8  (1008)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal3  (5001)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has te

로드 완료: qwen3-8b


MTL : EMBED_LIBRARY = 1 | CPU : NEON = 1 | ARM_FMA = 1 | FP16_VA = 1 | MATMUL_INT8 = 1 | DOTPROD = 1 | ACCELERATE = 1 | REPACK = 1 | 
Model metadata: {'general.quantization_version': '2', 'tokenizer.ggml.padding_token_id': '151643', 'qwen3.context_length': '40960', 'tokenizer.ggml.eos_token_id': '151645', 'tokenizer.ggml.pre': 'qwen2', 'tokenizer.ggml.model': 'gpt2', 'tokenizer.ggml.add_bos_token': 'false', 'general.name': 'Qwen3 8B Awq Compatible Instruct', 'qwen3.attention.value_length': '128', 'qwen3.attention.key_length': '128', 'qwen3.attention.layer_norm_rms_epsilon': '0.000001', 'qwen3.rope.freq_base': '1000000.000000', 'general.architecture': 'qwen3', 'qwen3.attention.head_count_kv': '8', 'tokenizer.chat_template': '{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0].role == \'system\' %}\n        {{- messages[0].content + \'\\n\\n\' }}\n    {%- endif %}\n    {{- "# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou a

In [ ]:
# ── 실험 루프 ─────────────────────────────────────
base_prompt = Path(f"prompts/{ROLE}_{MODE}_{SHOT}.txt").read_text(encoding="utf-8").strip()

candidates = ADJ_CANDIDATES if ROLE == "speaker" else STATE_CANDIDATES
outer_loop = STATE_VAR      if ROLE == "speaker" else ADJECTIVES

rows  = []
total = len(SITUATIONS) * len(RELATIONSHIP_VAR) * len(outer_loop) * REPETITIONS
print(f"총 {total} rows 생성 예정")
count = 0

for situation, scenario_template in SITUATIONS.items():
    for rel in RELATIONSHIP_VAR:
        for outer in outer_loop:
            for _ in range(REPETITIONS):
                personA, personB = sample(NAME_VARIATIONS, 2)
                scenario = scenario_template.format(personA=personA, personB=personB)
                thing    = THING_KEYWORDS[situation]

                fmt = dict(
                    personA=personA, personB=personB,
                    relationship=rel, scenario=scenario, thing=thing,
                )
                if ROLE == "speaker":
                    fmt["state"] = outer
                else:
                    fmt["adjective"] = f" {outer}"   # leading space for tokenization

                prompt = base_prompt.format(**fmt)
                
                # for qwen3, enable the thinking mode
                if MODE == "choice" and "qwen3" in MODEL:
                    prompt += "\n/no_think"

                row = dict(
                    situation=situation, relationship=rel,
                    personA=personA, personB=personB,
                )
                row["state" if ROLE == "speaker" else "adjective"] = outer

                if MODE == "logit":
                    logits, probs, logprobs = score_candidates(llm, prompt, candidates)
                    for c in candidates:
                        k = c.strip()
                        row[f"logit_{k}"]   = logits[c]
                        row[f"prob_{k}"]    = probs[c]
                        row[f"logprob_{k}"] = logprobs[c]
                    row["pred_top1"] = max(candidates, key=lambda c: probs[c]).strip()
                else:
                    resp = llm(prompt, max_tokens=30, temperature=0.0, top_k=1, top_p=0.9)
                    row["response_text"] = resp["choices"][0]["text"].strip()

                rows.append(row)
                count += 1
                if count % 100 == 0:
                    label = f"state={outer}" if ROLE == "speaker" else f"adj={outer}"
                    print(f"  [{count}/{total}] {situation} | {rel} | {label}")

Path("results").mkdir(exist_ok=True)
out = Path(f"results/{ROLE}_{MODE}_{MODEL}_{SHOT}.csv")
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

print(f"\n완료 → {out}  ({len(rows)} rows)")

총 1000 rows 생성 예정


In [21]:
# ── 실험 루프 ─────────────────────────────────────
base_prompt = Path(f"prompts/{ROLE}_{MODE}_{SHOT}.txt").read_text(encoding="utf-8").strip()

candidates = ADJ_CANDIDATES if ROLE == "speaker" else STATE_CANDIDATES
outer_loop = STATE_VAR      if ROLE == "speaker" else ADJECTIVES

rows  = []
total = len(SITUATIONS) * len(RELATIONSHIP_VAR) * len(outer_loop) * REPETITIONS
print(f"총 {total} rows 생성 예정")
count = 0

for situation, scenario_template in SITUATIONS.items():
    for rel in RELATIONSHIP_VAR:
        for outer in outer_loop:
            for _ in range(REPETITIONS):
                personA, personB = sample(NAME_VARIATIONS, 2)
                scenario = scenario_template.format(personA=personA, personB=personB)
                thing    = THING_KEYWORDS[situation]

                fmt = dict(
                    personA=personA, personB=personB,
                    relationship=rel, scenario=scenario, thing=thing,
                )
                if ROLE == "speaker":
                    fmt["state"] = outer
                else:
                    fmt["adjective"] = f" {outer}"   # leading space for tokenization

                prompt = base_prompt.format(**fmt)

                if MODE == "choice" and "qwen3" in MODEL:
                    prompt += "\n/no_think"

                row = dict(
                    situation=situation, relationship=rel,
                    personA=personA, personB=personB,
                )
                row["state" if ROLE == "speaker" else "adjective"] = outer

                if MODE == "logit":
                    logits, probs, logprobs = score_candidates(llm, prompt, candidates)
                    for c in candidates:
                        k = c.strip()
                        row[f"logit_{k}"]   = logits[c]
                        row[f"prob_{k}"]    = probs[c]
                        row[f"logprob_{k}"] = logprobs[c]
                    row["pred_top1"] = max(candidates, key=lambda c: probs[c]).strip()
                else:
                    resp = llm(prompt, max_tokens=30, temperature=0.0, top_k=1, top_p=0.9)
                    row["response_text"] = resp["choices"][0]["text"].strip()

                rows.append(row)
                count += 1
                if count % 100 == 0:
                    label = f"state={outer}" if ROLE == "speaker" else f"adj={outer}"
                    print(f"  [{count}/{total}] {situation} | {rel} | {label}")

Path("results").mkdir(exist_ok=True)
out = Path(f"results/{ROLE}_{MODE}_{MODEL}_{SHOT}_test.csv")
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

print(f"\n완료 → {out}  ({len(rows)} rows)")

총 100 rows 생성 예정


Llama.generate: 42 prefix-match hit, remaining 115 prompt tokens to eval
llama_perf_context_print:        load time =       1.88 ms
llama_perf_context_print: prompt eval time =    8434.01 ms /   116 tokens (   72.71 ms per token,    13.75 tokens per second)
llama_perf_context_print:        eval time =    2815.21 ms /    29 runs   (   97.08 ms per token,    10.30 tokens per second)
llama_perf_context_print:       total time =    9274.80 ms /   145 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 42 prefix-match hit, remaining 117 prompt tokens to eval
llama_perf_context_print:        load time =       1.88 ms
llama_perf_context_print: prompt eval time =    5166.75 ms /   117 tokens (   44.16 ms per token,    22.64 tokens per second)
llama_perf_context_print:        eval time =    2843.88 ms /    29 runs   (   98.06 ms per token,    10.20 tokens per second)
llama_perf_context_print:       total time =    8026.47 ms /   146 tokens
llama_perf_context_print:   

  [100/100] Gitarre | Gefürchtete Chefin | adj=schrecklich

완료 → results/listener_choice_qwen3-8b_zero_test.csv  (100 rows)
